In [ ]:
import torch
import torch.nn as nn

class HuberMapeFeatureLoss(nn.HuberLoss):
    def __init__(self, delta=1.0, reduction='mean', eps=1e-8):
        # Initialize the parent Huber Loss with 'none' to keep element-wise values
        super().__init__(reduction='none', delta=delta)
        self.user_reduction = reduction
        self.eps = eps # Safeguard against dividing by zero sales

    def forward(self, input: torch.Tensor, target: torch.Tensor, x_feature: torch.Tensor) -> torch.Tensor:
        """
        input: Model predictions (y_hat)
        target: Actual values (y)
        x_feature: Binary tensor (0 or 1) of the same shape as input/target
        """
        # 1. Calculate standard Huber Loss element-wise
        huber_base = super().forward(input, target)
        
        # 2. Calculate Absolute Percentage Error element-wise
        # Formula: |target - input| / max(|target|, eps)
        absolute_error = torch.abs(target - input)
        absolute_percentage_error = absolute_error / torch.clamp(torch.abs(target), min=self.eps)
        
        # 3. Multiply the percentage error by your binary feature x
        # If x is 0, this term becomes 0. If x is 1, it retains the error.
        mape_penalty = x_feature * absolute_percentage_error
        
        # 4. Add the penalty to the base Huber Loss
        # Total Loss = HuberLoss + (x * APE)
        total_loss = huber_base + mape_penalty
        
        # 5. Apply the user's requested reduction (mean or sum)
        if self.user_reduction == 'mean':
            return total_loss.mean()
        elif self.user_reduction == 'sum':
            return total_loss.sum()
        else:
            return total_loss